# Angular Gini and Gini in WID income data (Section 5.5)

This notebook reads the country files of the World Inequality Database (WID) bulk download, evaluates the indices **directly** (no intermediate files are written) and reproduces the numbers of Table 9 of the manuscript.

**Data.** `WID_data/WID_data_XX.csv` (one file per country, from the country pages of wid.world). Only five variables are kept in memory:
`aptinc992j` / `sptinc992j` (average and share of pre-tax national income, equal-split adults), `adiinc992j` / `sdiinc992j` (post-tax national income) and `iquali999i` (WID inequality transparency index).

**Method.** For each country and year the percentile groups are combined into the finest partition of the population without overlaps (99 percentiles and finer groups in the top percentile). Each group is represented by its average income, with its population share as weight. Because within-group inequality is ignored, all indices are *lower bounds*. Zero incomes at the bottom are admitted by the Gini and the Angular Gini; GE(0) is undefined if a group average is zero.

**Runtime.** About one minute for 15 countries and all years.

**Outputs** (folder `output/`): `ag_wid2024_indices_all_years.csv`, `ag_wid2024_main_table_pretax.csv`, `ag_wid2024_main_table_posttax.csv`, `ag_wid2024_discordance_<concept>.csv`.

**Requirements:** numpy, pandas, scipy.

In [1]:
import os, csv, re, time
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau

pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

INPUT_DIR  = 'WID_data'
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

COUNTRIES = ['AU','BR','DE','ES','FR','GB','IT','JP','NL','NO','PL','RU','SE','US','ZA']
MAIN      = ['ZA','BR','US','RU','JP','DE','FR','SE']          # countries of Table 9
NAMES     = {'ZA':'South Africa','BR':'Brazil','US':'United States','RU':'Russia',
             'JP':'Japan','DE':'Germany','FR':'France','SE':'Sweden'}
CONCEPTS  = {'pre-tax':  ('aptinc992j', 'sptinc992j'),
             'post-tax': ('adiinc992j', 'sdiinc992j')}
VARIABLES = ['aptinc992j','sptinc992j','adiinc992j','sdiinc992j','iquali999i']
YEAR      = 2024            # latest year available for all main countries
CHUNK     = 100_000

## 1. Index functions (weighted)

`gini_ag` computes the Gini coefficient and the Angular Gini exactly for weighted groups, summing over all pairs of groups.

In [2]:
def _ang(a, b):
    th = np.arctan2(a, b); ph = np.pi / 2 - th
    ph = np.where(ph > np.pi, ph - 2 * np.pi, ph)
    return (2 / np.pi) * np.abs(th - ph)           # angular differential, = (2/pi)|atan2(a,b) - atan2(b,a)|

def gini_ag(y, w, chunk=1200):
    y = np.asarray(y, float); w = np.asarray(w, float); n = len(y)
    W = w.sum(); mu = (w * y).sum() / W; sg = sa = 0.0
    for i0 in range(0, n, chunk):
        i1 = min(i0 + chunk, n); a = y[i0:i1, None]; wa = w[i0:i1, None]
        b = y[None, i0:i1]; ww = wa * w[None, i0:i1]; d = np.abs(a - b)
        sg += (ww * d).sum(); sa += (ww * d * _ang(a, b)).sum()
        if i1 < n:
            b = y[None, i1:]; ww = wa * w[None, i1:]; d = np.abs(a - b)
            sg += 2 * (ww * d).sum(); sa += 2 * (ww * d * _ang(a, b)).sum()
    den = 2 * W**2 * mu
    return sg / den, sa / den

def palma_w(y, w):
    o = np.argsort(y); y = y[o]; w = w[o]
    p  = np.concatenate([[0], np.cumsum(w)]) / w.sum()
    Lc = np.concatenate([[0], np.cumsum(w * y)]) / (w * y).sum()
    return (1 - np.interp(0.9, p, Lc)) / np.interp(0.4, p, Lc)

def ge1_w(y, w):
    if (y < 0).any(): return np.nan
    r = y / ((w * y).sum() / w.sum()); t = np.where(r > 0, r * np.log(np.where(r > 0, r, 1.0)), 0.0)
    return (w * t).sum() / w.sum()

def ge0_w(y, w):
    if (y <= 0).any(): return np.nan                 # undefined if a group average is zero
    r = y / ((w * y).sum() / w.sum()); return -(w * np.log(r)).sum() / w.sum()

def gastwirth_w(y, w):
    s = y[:, None] + y[None, :]; d = np.abs(y[:, None] - y[None, :])
    with np.errstate(divide='ignore', invalid='ignore'):
        g = np.where(s > 0, d / s, 0.0)
    return (w[:, None] * w[None, :] * g).sum() / w.sum()**2

def indices(y, w):
    g, a = gini_ag(y, w)
    return dict(Gini=g, AG=a, Palma=palma_w(y, w), GE1=ge1_w(y, w), GE0=ge0_w(y, w), Gastwirth=gastwirth_w(y, w))

# Check before any calculation: (-4, 5, 0) -> Gini 6.00, Angular Gini 11.24; (4, 8, 12, 36, 40) -> 0.400, 0.285
g, a = gini_ag([-4, 5, 0], np.ones(3));            assert abs(g - 6.00) < 5e-3 and abs(a - 11.24) < 5e-3, (g, a)
g2, a2 = gini_ag([4, 8, 12, 36, 40], np.ones(5));  assert abs(g2 - 0.400) < 5e-4 and abs(a2 - 0.285) < 5e-4, (g2, a2)
print(f'Check passed: (-4,5,0): Gini {g:.2f}, AG {a:.2f};  (4,8,12,36,40): Gini {g2:.3f}, AG {a2:.3f}')

Check passed: (-4,5,0): Gini 6.00, AG 11.24;  (4,8,12,36,40): Gini 0.400, AG 0.285


## 2. Read the WID files and keep only the five variables in memory

The files are read in chunks, because the original files are large. Nothing is written to disk.

In [3]:
frames = {}
for c in COUNTRIES:
    path = os.path.join(INPUT_DIR, f'WID_data_{c}.csv')
    if not os.path.exists(path):
        print(f'{c}: file missing'); continue
    with open(path, 'r', encoding='utf-8-sig') as fh:
        sample = fh.read(4096)
    try:    sep = csv.Sniffer().sniff(sample, delimiters=';,\t').delimiter
    except csv.Error: sep = ';'
    parts = []
    for ch in pd.read_csv(path, sep=sep, encoding='utf-8-sig', chunksize=CHUNK, low_memory=False):
        ch.columns = ch.columns.str.strip().str.lower()
        parts.append(ch[ch['variable'].isin(VARIABLES)])
    frames[c] = pd.concat(parts, ignore_index=True)
    print(f"{c}: {len(frames[c]):>6,} rows kept, variables: {sorted(frames[c].variable.unique())}")

AU: 126,942 rows kept, variables: ['adiinc992j', 'aptinc992j', 'iquali999i', 'sdiinc992j', 'sptinc992j']
BR: 78,186 rows kept, variables: ['adiinc992j', 'aptinc992j', 'iquali999i', 'sdiinc992j', 'sptinc992j']
DE: 130,044 rows kept, variables: ['adiinc992j', 'aptinc992j', 'iquali999i', 'sdiinc992j', 'sptinc992j']
ES: 95,214 rows kept, variables: ['adiinc992j', 'aptinc992j', 'iquali999i', 'sdiinc992j', 'sptinc992j']
FR: 123,852 rows kept, variables: ['adiinc992j', 'aptinc992j', 'iquali999i', 'sdiinc992j', 'sptinc992j']
GB: 127,722 rows kept, variables: ['adiinc992j', 'aptinc992j', 'iquali999i', 'sdiinc992j', 'sptinc992j']
IT: 82,830 rows kept, variables: ['adiinc992j', 'aptinc992j', 'iquali999i', 'sdiinc992j', 'sptinc992j']
JP: 143,976 rows kept, variables: ['adiinc992j', 'aptinc992j', 'iquali999i', 'sdiinc992j', 'sptinc992j']
NL: 107,587 rows kept, variables: ['adiinc992j', 'aptinc992j', 'iquali999i', 'sdiinc992j', 'sptinc992j']
NO: 113,005 rows kept, variables: ['adiinc992j', 'aptinc99

## 3. Partition into non-overlapping groups and calculation for every country and year

`tile` starts at 0 and always takes the group with the smallest upper bound; a country-year is used only if the partition is complete and no group average is missing.

In [4]:
PAT = re.compile(r'^p([\d.]+)p([\d.]+)$')
def bounds(p):
    m = PAT.match(p); return round(float(m.group(1)), 5), round(float(m.group(2)), 5)

def tile(pcts):
    by = {}
    for p in pcts:
        lo, hi = bounds(p)
        if hi > lo: by.setdefault(lo, []).append((hi, p))
    cur, out = 0.0, []
    while cur < 100 - 1e-9:
        if cur not in by: return None
        hi, p = min(by[cur]); out.append((cur, hi, p)); cur = hi
    return out

def group_table(d, avg, shr, year):
    a = d[(d.variable == avg) & (d.year == year)].set_index('percentile')
    s = d[(d.variable == shr) & (d.year == year)].set_index('percentile')
    if a.empty: return None
    t = tile(a.index)
    if t is None: return None
    g = pd.DataFrame([(lo, hi, p, (hi - lo) / 100, a.value.get(p, np.nan), s.value.get(p, np.nan)) for lo, hi, p in t],
                     columns=['lo', 'hi', 'pct', 'w', 'avg', 'share'])
    return None if g.avg.isna().any() else g

t0 = time.time(); rows = []
for c, d in frames.items():
    for conc, (av, sh) in CONCEPTS.items():
        for y in sorted(d[d.variable == av].year.unique()):
            g = group_table(d, av, sh, y)
            if g is None: continue
            yv, w = g.avg.values.astype(float), g.w.values
            r = dict(country=c, concept=conc, year=y, n_groups=len(g), sum_share=g.share.sum(),
                     mean=(w * yv).sum() / w.sum(), n_zero=int((yv == 0).sum()), n_neg=int((yv < 0).sum()))
            r.update(indices(yv, w))
            rows.append(r)
res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUTPUT_DIR, 'ag_wid2024_indices_all_years.csv'), index=False)
print(f'{len(res)} country-years computed in {time.time() - t0:.0f} s')
print('Sum of shares:', round(float(res.sum_share.min()), 4), 'to', round(float(res.sum_share.max()), 4),
      '| groups with negative average:', int(res.n_neg.sum()))

2211 country-years computed in 38 s
Sum of shares: 0.9988 to 1.001 | groups with negative average: 0


## 4. Main result: pre-tax national income, 2024 (Table 9)

In [5]:
iq = {c: frames[c].query("variable == 'iquali999i'").value.iloc[0] for c in MAIN if c in frames}
def main_table(concept):
    X = res[(res.concept == concept) & (res.year == YEAR)].set_index('country').loc[MAIN].sort_values('Gini', ascending=False)
    T = pd.DataFrame({'Gini': X.Gini, 'Angular Gini': X.AG, 'AG/Gini': X.AG / X.Gini, 'Palma': X.Palma,
                      'GE(1)': X.GE1, 'GE(0)': X.GE0, 'Gastwirth': X.Gastwirth,
                      'Rank Gini': X.Gini.rank(ascending=False).astype(int),
                      'Rank AG': X.AG.rank(ascending=False).astype(int),
                      'Zero groups': X.n_zero, 'Transp.': pd.Series(iq)})
    T.index = [NAMES[c] for c in T.index]
    return T

T_pre = main_table('pre-tax')
T_pre.to_csv(os.path.join(OUTPUT_DIR, 'ag_wid2024_main_table_pretax.csv'))
T_pre

,Gini,Angular Gini,AG/Gini,Palma,GE(1),GE(0),Gastwirth,Rank Gini,Rank AG,Zero groups,Transp.
South Africa,0.7615,0.6799,0.8929,20.7145,1.3422,NaN,0.6014,1,1,5,3.8000
Brazil,0.6948,0.6035,0.8685,11.0140,1.4173,1.6760,0.5757,2,2,0,5.5000
United States,0.5869,0.4697,0.8003,5.4731,0.9709,0.7414,0.4861,3,4,0,15.4000
Russia,0.5820,0.4738,0.8142,4.8387,1.0135,NaN,0.4484,4,3,5,4.5000
Japan,0.5144,0.4022,0.7820,3.9177,0.6003,NaN,0.4552,5,5,5,6.0000
Germany,0.4901,0.3629,0.7406,3.0797,0.5686,NaN,0.4521,6,6,5,12.9000
France,0.4576,0.3296,0.7202,2.6319,0.5072,NaN,0.4405,7,7,5,15.1000
Sweden,0.4208,0.2938,0.6983,2.0821,0.4452,NaN,0.4069,8,8,5,14.3000


In [6]:
# Rows for the LaTeX table of the manuscript
for name, r in T_pre.iterrows():
    print(f"{name} & {r['Gini']:.3f} & {r['Angular Gini']:.3f} & {r['AG/Gini']:.3f} & {r['Palma']:.1f} & {r['GE(1)']:.3f} & "
          f"{int(r['Rank Gini'])} & {int(r['Rank AG'])} & {int(r['Zero groups'])} & {r['Transp.']:.1f} \\\\")

South Africa & 0.761 & 0.680 & 0.893 & 20.7 & 1.342 & 1 & 1 & 5 & 3.8 \\
Brazil & 0.695 & 0.603 & 0.868 & 11.0 & 1.417 & 2 & 2 & 0 & 5.5 \\
United States & 0.587 & 0.470 & 0.800 & 5.5 & 0.971 & 3 & 4 & 0 & 15.4 \\
Russia & 0.582 & 0.474 & 0.814 & 4.8 & 1.013 & 4 & 3 & 5 & 4.5 \\
Japan & 0.514 & 0.402 & 0.782 & 3.9 & 0.600 & 5 & 5 & 5 & 6.0 \\
Germany & 0.490 & 0.363 & 0.741 & 3.1 & 0.569 & 6 & 6 & 5 & 12.9 \\
France & 0.458 & 0.330 & 0.720 & 2.6 & 0.507 & 7 & 7 & 5 & 15.1 \\
Sweden & 0.421 & 0.294 & 0.698 & 2.1 & 0.445 & 8 & 8 & 5 & 14.3 \\


Post-tax national income (series `adiinc992j`), same countries and year:

In [7]:
T_post = main_table('post-tax'); T_post.to_csv(os.path.join(OUTPUT_DIR, 'ag_wid2024_main_table_posttax.csv')); T_post

,Gini,Angular Gini,AG/Gini,Palma,GE(1),GE(0),Gastwirth,Rank Gini,Rank AG,Zero groups,Transp.
Brazil,0.6478,0.5423,0.8372,7.0274,1.0390,0.7787,0.4428,2,2,0,5.5000
Germany,0.3674,0.2323,0.6321,1.6243,0.3480,0.2467,0.3048,6,6,0,12.9000
France,0.2776,0.1443,0.5197,1.0235,0.2066,0.1362,0.2365,7,7,0,15.1000
Japan,0.4310,0.3080,0.7147,2.4543,0.4211,0.3679,0.3600,5,5,0,6.0000
Russia,0.5660,0.4532,0.8008,4.3931,0.9695,0.6186,0.4093,3,3,0,4.5000
Sweden,0.2543,0.1299,0.5110,0.9020,0.1803,0.1257,0.2295,8,8,0,14.3000
United States,0.4673,0.3375,0.7222,2.6534,0.6147,0.4594,0.3950,4,4,0,15.4000
South Africa,0.6513,0.5429,0.8335,6.9675,0.9977,0.7650,0.4313,1,1,0,3.8000


## 5. Do the rankings differ between the Gini coefficient and the Angular Gini?

First, rank correlations and the countries whose rank changes, for all 15 countries in 2024. Second, the share of country pairs that the two indices order differently, in every year with at least eight countries.

In [8]:
def rank_report(concept, year=YEAR):
    X = res[(res.concept == concept) & (res.year == year)].set_index('country')
    out = []
    for other, lab in [('AG', 'Angular Gini'), ('Palma', 'Palma'), ('GE1', 'GE(1)')]:
        out.append(dict(pair=f'Gini vs {lab}', spearman=spearmanr(X.Gini, X[other])[0], kendall=kendalltau(X.Gini, X[other])[0]))
    out.append(dict(pair='Angular Gini vs GE(1)', spearman=spearmanr(X.AG, X.GE1)[0], kendall=kendalltau(X.AG, X.GE1)[0]))
    r1, r2 = X.Gini.rank(ascending=False), X.AG.rank(ascending=False)
    changed = {c: f'{int(r1[c])} -> {int(r2[c])}' for c in X.index if r1[c] != r2[c]}
    return pd.DataFrame(out), changed, len(X)

for concept in CONCEPTS:
    R, changed, n = rank_report(concept)
    print(f'{concept}, {YEAR}, {n} countries; rank changes Gini -> Angular Gini: {changed or "none"}')
    display(R)

pre-tax, 2024, 15 countries; rank changes Gini -> Angular Gini: {'RU': '4 -> 3', 'US': '3 -> 4'}


,pair,spearman,kendall
0,Gini vs Angular Gini,0.9964,0.9810
1,Gini vs Palma,1.0000,1.0000
2,Gini vs GE(1),0.9250,0.7905
3,Angular Gini vs GE(1),0.9286,0.8095


post-tax, 2024, 15 countries; rank changes Gini -> Angular Gini: none


,pair,spearman,kendall
0,Gini vs Angular Gini,1.0000,1.0000
1,Gini vs Palma,0.9964,0.9810
2,Gini vs GE(1),0.9679,0.8667
3,Angular Gini vs GE(1),0.9679,0.8667


In [9]:
def discordance(concept):
    out = []
    for y, Y in res[res.concept == concept].groupby('year'):
        if len(Y) < 8: continue
        g, a = Y.Gini.values, Y.AG.values; n = len(Y)
        disc = sum((g[i] - g[j]) * (a[i] - a[j]) < 0 for i in range(n) for j in range(i + 1, n))
        out.append(dict(year=y, countries=n, share_discordant_pairs=disc / (n * (n - 1) / 2), kendall_tau=kendalltau(g, a)[0]))
    return pd.DataFrame(out)

D = {k: discordance(k) for k in CONCEPTS}
for k, v in D.items():
    v.to_csv(os.path.join(OUTPUT_DIR, f'ag_wid2024_discordance_{k}.csv'), index=False)
    print(f"{k}: {len(v)} years ({v.year.min()}-{v.year.max()}); mean share of discordant country pairs "
          f"{100*v.share_discordant_pairs.mean():.1f} %, maximum {100*v.share_discordant_pairs.max():.1f} %, minimum tau {v.kendall_tau.min():.3f}")
D['pre-tax'][D['pre-tax'].year.isin([1980, 1990, 2000, 2010, 2020, 2024])]

pre-tax: 109 years (1820-2024); mean share of discordant country pairs 2.2 %, maximum 8.3 %, minimum tau 0.833
post-tax: 45 years (1980-2024); mean share of discordant country pairs 1.3 %, maximum 6.7 %, minimum tau 0.867


,year,countries,share_discordant_pairs,kendall_tau
64,1980,15,0.0381,0.9238
74,1990,15,0.0190,0.9619
84,2000,15,0.0000,1.0000
94,2010,15,0.0000,1.0000
104,2020,15,0.0095,0.9810
108,2024,15,0.0095,0.9810


## 6. Lower bounds: sensitivity to the coarseness of the grouping (2024, pre-tax)

The same indices from the 127 groups, from the 100 percentiles (top percentile as one group) and from the 10 deciles. Coarser groups ignore more of the inequality within groups, so the values fall.

In [10]:
rows = []
for c, d in frames.items():
    av, sh = CONCEPTS['pre-tax']
    g = group_table(d, av, sh, YEAR)
    a = d[(d.variable == av) & (d.year == YEAR)].set_index('percentile').value
    parts = {'127 groups': (g.avg.values, g.w.values),
             '100 percentiles': (a.loc[[f'p{i}p{i+1}' for i in range(100)]].values, np.full(100, 0.01)),
             '10 deciles': (a.loc[[f'p{10*i}p{10*i+10}' for i in range(10)]].values, np.full(10, 0.1))}
    for lab, (y, w) in parts.items():
        gi, ag = gini_ag(y, w); rows.append(dict(country=c, grouping=lab, Gini=gi, AG=ag))
P = pd.DataFrame(rows).pivot(index='country', columns='grouping', values=['Gini', 'AG'])
chg = pd.DataFrame({(ix, lab): 100 * (P[ix][lab] / P[ix]['127 groups'] - 1) for ix in ['Gini', 'AG'] for lab in ['100 percentiles', '10 deciles']})
print('Change relative to the 127 groups, in percent:')
display(chg.round(2))
print(chg.agg(['min', 'max']).round(2))
for lab in ['127 groups', '100 percentiles', '10 deciles']:
    r1, r2 = P['Gini'][lab].rank(ascending=False), P['AG'][lab].rank(ascending=False)
    print(f"{lab}: Spearman {spearmanr(P['Gini'][lab], P['AG'][lab])[0]:.3f}; rank changes:", {c: f'{int(r1[c])} -> {int(r2[c])}' for c in r1.index if r1[c] != r2[c]} or 'none')

Change relative to the 127 groups, in percent:


Gini                         AG           
        100 percentiles 10 deciles 100 percentiles 10 deciles
country                                                      
AU              -0.0700    -2.8800         -0.2200    -4.5900
BR              -0.1500    -4.7200         -0.2200    -5.2200
DE              -0.1100    -3.4400         -0.2700    -5.6100
ES              -0.1200    -3.4000         -0.3600    -6.5800
FR              -0.1100    -3.2500         -0.3300    -6.0500
GB              -0.1200    -3.4600         -0.3100    -6.0600
IT              -0.0700    -2.8100         -0.2500    -4.8600
JP              -0.0300    -3.6500         -0.0400    -4.8500
NL              -0.0300    -2.3400         -0.1500    -3.7100
NO              -0.0900    -2.9400         -0.3700    -6.0200
PL              -0.1500    -3.7600         -0.3800    -6.6600
RU              -0.2000    -4.7800         -0.3000    -6.2500
SE              -0.1100    -3.2900         -0.3800    -6.8700
US              -0.1800    -4.1800         -0.3400    -5.8300
ZA              -0.0800    -4.0100         -0.1200    -3.3800

               Gini                         AG           
    100 percentiles 10 deciles 100 percentiles 10 deciles
min         -0.2000    -4.7800         -0.3800    -6.8700
max         -0.0300    -2.3400         -0.0400    -3.3800
127 groups: Spearman 0.996; rank changes: {'RU': '4 -> 3', 'US': '3 -> 4'}
100 percentiles: Spearman 0.996; rank changes: {'RU': '4 -> 3', 'US': '3 -> 4'}
10 deciles: Spearman 0.996; rank changes: {'RU': '4 -> 3', 'US': '3 -> 4'}


## 7. Zero incomes and negative values

The Angular Gini remains defined if group averages are zero; GE(0) does not. No group average is negative, so a floor at zero would change nothing.

In [11]:
Z = res[(res.concept == 'pre-tax') & (res.year == YEAR)].set_index('country')[['n_zero', 'n_neg', 'GE0']]
Z['GE(0) defined'] = Z.GE0.notna()
display(Z)
print('Country-years with a negative group average:', int((res.n_neg > 0).sum()), 'of', len(res))

,n_zero,n_neg,GE0,GE(0) defined
country,,,,
AU,5,0,NaN,False
BR,0,0,1.6760,True
DE,5,0,NaN,False
ES,5,0,NaN,False
FR,5,0,NaN,False
GB,5,0,NaN,False
IT,5,0,NaN,False
JP,5,0,NaN,False
NL,5,0,NaN,False


Country-years with a negative group average: 0 of 2211
